# Bonus 07 — LlamaIndex as a data framework

LlamaIndex organizes the data path around an LLM application: **Documents** become **Nodes**, transformations enrich them, an **Index** connects storage to retrieval, a **QueryEngine** combines retrieval with synthesis, and the whole engine can become an agent tool.

You will build a local policy index, inspect metadata and embeddings, compare semantic retrieval with deterministic metadata filtering, synthesize an answer with sources, persist and reload the index, expose it as a tool, and account for tokens and cost.

> LlamaIndex is not a vector database and retrieval is not automatically an agent. It is a framework of composable data abstractions.

## 1. Learn — follow the data, not the marketing name

```mermaid
flowchart LR
    F["Policy files"] --> D["Documents + trusted metadata"]
    D --> P["IngestionPipeline"]
    P --> S["SentenceSplitter"]
    S --> N["Nodes + relationships"]
    N --> E["Embedding transformation"]
    E --> V["VectorStoreIndex"]
    V --> R["Retriever"]
    R --> Q["QueryEngine + response synthesis"]
    Q --> O["Response + source nodes"]
    Q --> T["QueryEngineTool"]
    T --> A["FunctionAgent"]
```

Every arrow is a boundary you can inspect or replace. A five-line quickstart is convenient only after you understand which defaults those five lines select.

### The core objects

| Object | Responsibility | Important question |
|---|---|---|
| `Document` | source text plus metadata and exclusions | What data may be embedded or sent to an LLM? |
| `Node` | retrieval-sized text, metadata, relationships, optional embedding | What exactly is the searchable unit? |
| transformation | maps Documents or Nodes to new Nodes | How were chunks and embeddings produced? |
| `StorageContext` | document, index, graph, and vector stores | What is persisted, and where? |
| `VectorStoreIndex` | connects nodes to vector retrieval | Which embedding and storage defaults are active? |
| retriever | returns scored source nodes | Which candidates entered model context? |
| query engine | retrieval plus response synthesis | Did the LLM answer only from those nodes? |
| tool / agent | makes the query engine model-selectable | Was agentic choice actually needed? |

### Core versus integrations

Modern LlamaIndex is modular. This lab installs `llama-index-core`, `llama-index-embeddings-openai`, and `llama-index-llms-openai` rather than the umbrella package. Readers, vector stores, rerankers, observability systems, and other providers are separate integrations.

That integration ecosystem is useful for data engineers, but each connector is a dependency and a data-egress decision. Install only what the design uses.

### Classify before embedding

The course corpus also contains support-ticket examples and a planted hostile instruction. This lab sends **only the 20 customer-facing `policy_*.md` files** to the embedding service. Ticket-like material never enters this process.

A hosted embedding call is data egress. In production, classification, redaction, region, retention, contractual controls, and a local embedding option are architecture decisions—not notebook cleanup. Metadata filters narrow retrieval; they do not sanitize unsafe content.

### Use the lab's isolated environment

From `bonus/07_llamaindex_data_framework/`, run:

```bash
uv sync --locked
```

Then choose `bonus/07_llamaindex_data_framework/.venv/bin/python` as the VS Code kernel. The modular stack still resolves 94 installed distributions and OpenAI 2.54.0, so it must not change the core course environment.

## 2. Do — build the data path explicitly

In [ ]:
import json
import os
from importlib.metadata import distributions, version
from pathlib import Path
from tempfile import TemporaryDirectory

from dotenv import find_dotenv, load_dotenv
from llama_index.core import (
    Document,
    Settings,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage,
)
from llama_index.core.agent.workflow import (
    AgentOutput,
    FunctionAgent,
    ToolCall,
    ToolCallResult,
)
from llama_index.core.callbacks import CallbackManager, TokenCountingHandler
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode
from llama_index.core.tools import QueryEngineTool
from llama_index.core.vector_stores import (
    FilterOperator,
    MetadataFilter,
    MetadataFilters,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAIResponses

env_path = find_dotenv(usecwd=True)
assert env_path, "Repository-root .env not found"
load_dotenv(env_path)
ROOT = Path(env_path).parent
MODEL_DEFAULT = os.environ["MODEL_DEFAULT"]
EMBEDDING_MODEL = os.environ["EMBEDDING_MODEL"]
PRICE_INPUT = float(os.environ["PRICE_INPUT_PER_MILLION"])
PRICE_OUTPUT = float(os.environ["PRICE_OUTPUT_PER_MILLION"])

print("LlamaIndex core:", version("llama-index-core"))
print("OpenAI integration:", version("llama-index-llms-openai"))
print("Embedding integration:", version("llama-index-embeddings-openai"))
print("OpenAI SDK:", version("openai"))
print("Installed distributions:", len(list(distributions())))
print("Models from .env:", MODEL_DEFAULT, "/", EMBEDDING_MODEL)

### Configure the framework defaults once—and name the tradeoff

`Settings` propagates the model, embedding model, and callback manager into indexes, query engines, and tools. That is convenient integration glue. It is also global mutable state: tests and concurrent applications should isolate or reset it deliberately.

This lab uses the OpenAI Responses integration. Its correct output limit is `max_output_tokens`; reasoning options suppress unsupported sampling fields for this model family. No model name or price is hard-coded.

In [ ]:
token_counter = TokenCountingHandler()
callback_manager = CallbackManager([token_counter])

embed_model = OpenAIEmbedding(
    model=EMBEDDING_MODEL,
    callback_manager=callback_manager,
)
llm = OpenAIResponses(
    model=MODEL_DEFAULT,
    reasoning_options={"effort": "none"},
    max_output_tokens=500,
    store=False,
    callback_manager=callback_manager,
)

Settings.callback_manager = callback_manager
Settings.embed_model = embed_model
Settings.llm = llm

### Load Documents with trusted metadata

The filename determines the topic because the instructor controls this directory and naming convention. We explicitly choose which metadata reaches embeddings and which reaches the response-synthesis prompt.

In [ ]:
policy_paths = sorted((ROOT / "data/corpus").glob("policy_*.md"))
assert len(policy_paths) == 20
assert all("ticket_" not in path.name for path in policy_paths)

documents = [
    Document(
        text=path.read_text(),
        id_=path.stem,
        metadata={
            "file_name": path.name,
            "source_type": "customer_policy",
            "topic": path.stem.removeprefix("policy_"),
        },
        excluded_embed_metadata_keys=["file_name", "source_type"],
        excluded_llm_metadata_keys=["source_type"],
    )
    for path in policy_paths
]

print("Documents:", len(documents))
print("First ID:", documents[0].doc_id)
print("First metadata:", documents[0].metadata)
print("Embedded view:", documents[0].get_content(metadata_mode=MetadataMode.EMBED)[:140])
print("LLM view:", documents[0].get_content(metadata_mode=MetadataMode.LLM)[:180])

### An ingestion pipeline is an ordered transformation list

`SentenceSplitter` creates retrieval-sized Nodes and source relationships. `OpenAIEmbedding` then attaches a vector to each Node. We call the asynchronous path because frameworks increasingly compose network transformations concurrently.

In [ ]:
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=180, chunk_overlap=20),
        embed_model,
    ]
)
nodes = await pipeline.arun(documents=documents, show_progress=False)
ingestion_embedding_tokens = token_counter.total_embedding_token_count

print("Nodes:", len(nodes))
print("Embedding tokens (local tokenizer estimate):", ingestion_embedding_tokens)

### Inspect a Node before indexing it

A Node is not just a string. It has a stable ID, source relationship, metadata views, and an embedding. Those fields drive retrieval, provenance, persistence, and prompt construction.

In [ ]:
sample_node = nodes[0]
print("Node ID:", sample_node.node_id)
print("Source document ID:", sample_node.ref_doc_id)
print("Metadata:", sample_node.metadata)
print("Relationships:", [relationship.value for relationship in sample_node.relationships])
print("Embedding dimensions:", len(sample_node.embedding or []))
print("Text preview:", sample_node.text[:120])
assert len(sample_node.embedding or []) == 1536

### Build an index without pretending it is a database

`VectorStoreIndex` uses the default local `SimpleVectorStore` here. The index coordinates nodes, stores, and retrieval. A production vector database is a replaceable storage integration—not what makes the application LlamaIndex.

In [ ]:
policy_index = VectorStoreIndex(nodes=nodes)
print("Vector store:", type(policy_index.vector_store).__name__)
print("Indexed nodes:", len(policy_index.index_struct.nodes_dict))

### Retrieval returns evidence, not an answer

A retriever embeds the query, compares it with stored vectors, and returns `NodeWithScore` objects. No synthesis model is called yet.

In [ ]:
retriever = policy_index.as_retriever(similarity_top_k=3)
retrieved = await retriever.aretrieve(
    "same album arrived damaged twice and refund timing"
)
retrieval_rows = [
    {
        "file": item.node.metadata["file_name"],
        "score": round(item.score or 0, 3),
        "preview": item.node.text[:90].replace("\n", " "),
    }
    for item in retrieved
]
retrieval_rows

### Metadata filtering is deterministic narrowing

Semantic similarity answers “what sounds relevant?” A trusted metadata filter answers “which namespace is allowed?” Combine them when the application already knows a tenant, document type, region, version, or topic. Do not ask an embedding to rediscover a fact your software already knows.

In [ ]:
student_discount_filters = MetadataFilters(
    filters=[
        MetadataFilter(
            key="topic",
            value="student_discount",
            operator=FilterOperator.EQ,
        )
    ]
)
filtered_retriever = policy_index.as_retriever(
    similarity_top_k=3,
    filters=student_discount_filters,
)
filtered_nodes = await filtered_retriever.aretrieve(
    "Can the discount be added after an invoice?"
)
[(item.node.metadata["file_name"], round(item.score or 0, 3)) for item in filtered_nodes]

### A QueryEngine adds response synthesis

The engine retrieves top Nodes, formats them into a framework prompt, asks the LLM to synthesize, and returns a `Response` that retains its source nodes. Always inspect both answer and evidence.

In [ ]:
policy_query_engine = policy_index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact",
)
main_response = await policy_query_engine.aquery(
    "A customer received the same album damaged twice. What should support offer, "
    "and how long does the money take after approval?"
)
print(str(main_response))

In [ ]:
source_rows = [
    {
        "file": item.node.metadata["file_name"],
        "score": round(item.score or 0, 3),
        "text": item.node.text[:120].replace("\n", " "),
    }
    for item in main_response.source_nodes
]
print(json.dumps(source_rows, indent=2))

main_text = str(main_response).lower()
source_files = {row["file"] for row in source_rows}
assert "refund" in main_text
assert "5 to 10 business days" in main_text
assert "policy_damaged_media.md" in source_files
assert "policy_refunds.md" in source_files

### Persistence saves stores, not a magical index object

The local `StorageContext` writes its document store, index store, graph store, and vector stores. Loading recreates an index over those artifacts and still needs the embedding model for new queries. We use a temporary directory so the lab leaves no generated index in the repository.

In [ ]:
with TemporaryDirectory(prefix="llamaindex-policy-") as persist_dir:
    policy_index.set_index_id("support-policies")
    policy_index.storage_context.persist(persist_dir=persist_dir)
    persisted_files = sorted(path.name for path in Path(persist_dir).iterdir())

    loaded_storage = StorageContext.from_defaults(persist_dir=persist_dir)
    loaded_index = load_index_from_storage(
        loaded_storage,
        index_id="support-policies",
    )
    loaded_nodes = await loaded_index.as_retriever(
        similarity_top_k=1,
        filters=student_discount_filters,
    ).aretrieve("student discount on physical items")

print("Persisted files:", persisted_files)
print("Reloaded top source:", loaded_nodes[0].node.metadata["file_name"])
assert loaded_nodes[0].node.metadata["file_name"] == "policy_student_discount.md"

## 3. Observe — turn a data engine into a tool

A QueryEngine is useful without an agent. Wrapping it in `QueryEngineTool` is justified only when a model must choose among tools or decide whether retrieval is needed. The tool call adds an outer agent loop around an inner retrieval-and-synthesis call.

In [ ]:
policy_tool = QueryEngineTool.from_defaults(
    query_engine=policy_query_engine,
    name="query_support_policies",
    description=(
        "Use for exact questions about shop returns, refunds, shipping, discounts, "
        "digital purchases, and support policies."
    ),
)
policy_agent = FunctionAgent(
    name="SupportPolicyAgent",
    description="Answers shop-policy questions using the indexed policy source.",
    system_prompt=(
        "For every shop-policy question, call query_support_policies before answering. "
        "Keep the answer grounded in that tool result."
    ),
    tools=[policy_tool],
    llm=llm,
    streaming=False,
)

In [ ]:
agent_handler = policy_agent.run(
    user_msg="A student already paid for a digital album. Can support apply the student discount now?"
)
agent_events = []

async for event in agent_handler.stream_events():
    if isinstance(event, ToolCall):
        agent_events.append({
            "event": "ToolCall",
            "tool": event.tool_name,
            "arguments": event.tool_kwargs,
        })
    elif isinstance(event, ToolCallResult):
        agent_events.append({
            "event": "ToolCallResult",
            "tool": event.tool_name,
            "content": str(event.tool_output.content),
        })
    elif isinstance(event, AgentOutput):
        agent_events.append({
            "event": "AgentOutput",
            "tool_calls": [call.tool_name for call in event.tool_calls],
            "content": event.response.content,
        })

agent_result = await agent_handler
print(agent_result.response.content)

### Events show the outer loop; source nodes show the inner evidence

The agent proposes a query-engine tool call, the tool runs retrieval plus synthesis, and the agent writes a final response. Count both layers. A fluent outer answer does not replace source inspection inside the query-engine result.

In [ ]:
print(json.dumps(agent_events, indent=2, default=str))

tool_calls = [event for event in agent_events if event["event"] == "ToolCall"]
tool_results = [event for event in agent_events if event["event"] == "ToolCallResult"]
assert len(tool_calls) == 1
assert len(tool_results) == 1
assert tool_calls[0]["tool"] == "query_support_policies"
assert "no" in (agent_result.response.content or "").lower()

prompt_tokens = token_counter.prompt_llm_token_count
completion_tokens = token_counter.completion_llm_token_count
llm_requests = len(token_counter.llm_token_counts)
estimated_llm_cost = (
    prompt_tokens * PRICE_INPUT + completion_tokens * PRICE_OUTPUT
) / 1_000_000

print("LLM requests observed:", llm_requests)
print("LLM prompt tokens:", prompt_tokens)
print("LLM completion tokens:", completion_tokens)
print("Embedding tokens (local tokenizer estimate):", ingestion_embedding_tokens)
print(f"Estimated LLM cost, assuming uncached input: ${estimated_llm_cost:.6f}")

### What LlamaIndex bought—and what it did not

It bought common data objects, transformation composition, metadata-aware retrieval, swappable stores, source-carrying responses, persistence conventions, query-engine composition, tool adapters, workflows, and integrations. It did **not** classify data, select a safe chunking strategy, make metadata trustworthy, evaluate retrieval, prevent prompt injection, authorize actions, or make an agent necessary.

For teams that want control, use Documents, Nodes, retrievers, or stores selectively. For teams that want speed, use the higher-level query engine and tool adapters—but keep the evidence and defaults observable.

## 4. Challenge — build a topic-filtered gift-card engine

Create a `MetadataFilters` object for topic `gift_cards`. Build a query engine over `policy_index` with that filter and ask:

> Do gift cards expire, and can a customer exchange one for cash?

Acceptance criteria:

- the response is not `None`;
- every source node is `policy_gift_cards.md`;
- the answer says gift cards do not expire;
- cash exchange is allowed only where law requires it.

Do not create an agent. The application already knows which topic to query, so a deterministic filtered QueryEngine is the simpler design.

In [ ]:
# TODO: create gift_card_filters, gift_card_engine, and await its query.
gift_card_filters = None
gift_card_engine = None
challenge_response = None

In [ ]:
assert challenge_response is not None, "Run the filtered query engine"
challenge_sources = {
    item.node.metadata["file_name"] for item in challenge_response.source_nodes
}
challenge_text = str(challenge_response).lower()
assert challenge_sources == {"policy_gift_cards.md"}
assert "do not expire" in challenge_text or "don't expire" in challenge_text
assert "cash" in challenge_text
assert "law" in challenge_text
print("Challenge passed")
print("Sources:", challenge_sources)
print(str(challenge_response))

## Takeaway

LlamaIndex is strongest when you treat it as a toolbox for data-aware LLM systems: control the Documents, inspect the Nodes, compose transformations, narrow with trusted metadata, retrieve before synthesizing, retain sources, persist known stores, and promote a QueryEngine to an agent tool only when model-directed choice adds value.